# 선택 실행 — TabICLv2 비교
기본 프로그램에 필요하지 않은 **추가 모델**입니다. 먼저 [선택 모델 설치 안내](../docs/선택모델_TabICL.md)를 따라 별도 패키지와 체크포인트를 준비하세요.

이 모델은 미리 학습된 표 데이터 모델입니다. 이번 데이터에서는 마스터를 기준으로 비교합니다. 실제 회사 데이터의 독립 진성 검출 성능이 입증된 것은 아닙니다.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path(os.environ.get("ECT_PROJECT_ROOT", str(Path.cwd())))
if not (ROOT / "ectlab").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "ectlab").is_dir():
    raise RuntimeError("프로젝트의 notebooks 폴더에서 열어 주세요.")
sys.path.insert(0, str(ROOT))
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
from IPython.display import display, HTML, FileLink
import pandas as pd
from ectlab.runtime import data_path
from ectlab.data import load_csv, inventory
from ectlab.pipeline import Settings, prepare, distance_analysis, train_models, summarize, export_run
from ectlab.plots import pca_plot, comparison_plot, true_case_plot, frequency_plot
print("실행 준비 완료. 다음 칸을 실행하세요.")

In [ ]:
DATA_MODE = os.environ.get("ECT_DATA_MODE", "demo")
DATA_PATH = data_path(ROOT, DATA_MODE, os.environ.get("ECT_PRODUCT", "C10A_49513-75270R"))
CHECKPOINT = ROOT / "checkpoints/tabicl-classifier-v2-20260212.ckpt"
df = load_csv(DATA_PATH)
settings = Settings()
packs, counts = prepare(df, settings)
display(HTML(f"<h3>{'연습용 합성 데이터' if DATA_MODE == 'demo' else '실제 회사 데이터'}</h3>"))
display(counts)

## 실행
기본 CPU 실행입니다. 인터넷으로 회사 데이터를 전송하는 과정은 없습니다. 사전에 받은 체크포인트를 로컬에서 읽습니다.

In [ ]:
result = train_models(packs, names=("TabICLv2",), experiments=("master_proxy",), checkpoint=CHECKPOINT)
display(summarize(result["predictions"]))
empty_distance = {"predictions": pd.DataFrame(), "details": pd.DataFrame(), "models": {}, "notes": []}
figures = {"TabICLv2_진성행_점수": true_case_plot(result["predictions"])}
display(figures["TabICLv2_진성행_점수"])
saved_folder = export_run(ROOT / "results", DATA_PATH, settings, packs, empty_distance, result, figures)